# 03. Análisis ético y equidad descriptiva

Este análisis mide diferencias de desempeño por sexo, raza y rango de edad en el test reservado. Son indicadores descriptivos sobre una fuente censal de 1994; no prueban causalidad ni determinan por sí solos si una decisión es justa. El modelo no debe usarse para decisiones de empleo, crédito, seguros u otros contextos de alto impacto.


In [1]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "utils.py").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from src.utils import (
    RANDOM_STATE, cargar_datos, construir_pipeline_completo, crear_grupo_edad,
    diferencia_max_min, identificar_columnas, metricas_por_grupo,
    obtener_features_y_objetivo,
)


## 1. Reproducir la evaluación

Se conserva la misma semilla, división estratificada y pipeline usados en el notebook 02. Los atributos sensibles se retienen en `X_test` para desagregar métricas, aunque también forman parte de los predictores en este ejercicio académico.


In [2]:
df = cargar_datos()
X, y = obtener_features_y_objetivo(df)
numericas, categoricas = identificar_columnas(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
modelo = construir_pipeline_completo(numericas, categoricas, "logistica")
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)


## 2. Métricas por sexo y raza

Se reportan tamaño de grupo, tasa positiva observada y predicha, recall, precision y F1. Los grupos pequeños producen estimaciones más inestables; se deben mostrar junto a su tamaño.


In [3]:
tablas = {}
for columna, archivo in [("sex", "metricas_por_sexo.csv"), ("race", "metricas_por_raza.csv")]:
    tabla_grupo = metricas_por_grupo(X_test, y_test, y_pred, columna)
    tabla_grupo.to_csv(ROOT / "data" / archivo, index=False, encoding="utf-8-sig")
    tablas[columna] = tabla_grupo
    print(f"Métricas por {columna}")
    display(tabla_grupo)


Métricas por sex


,Grupo,N,Tasa_Positivos_Real,Tasa_Positivos_Predicha,Recall,Precision,F1-Score,Positivos_Reales
0,Female,3289,0.1116,0.1396,0.6839,0.5468,0.6077,367
1,Male,6480,0.3042,0.4654,0.8656,0.5656,0.6842,1971


Métricas por race


,Grupo,N,Tasa_Positivos_Real,Tasa_Positivos_Predicha,Recall,Precision,F1-Score,Positivos_Reales
0,Amer-Indian-Eskimo,104,0.0865,0.2019,0.6667,0.2857,0.4000,9
1,Asian-Pac-Islander,304,0.2566,0.4046,0.8462,0.5366,0.6567,78
2,Black,944,0.1282,0.1515,0.6529,0.5524,0.5985,121
3,Other,69,0.1594,0.1884,0.6364,0.5385,0.5833,11
4,White,8348,0.2538,0.3803,0.8490,0.5666,0.6796,2119


## 3. Métricas por rango de edad


In [4]:
X_test_edad = X_test.copy()
X_test_edad["rango_edad"] = X_test_edad["age"].map(crear_grupo_edad)
tablas["age"] = metricas_por_grupo(X_test_edad, y_test, y_pred, "rango_edad")
tablas["age"].to_csv(ROOT / "data" / "metricas_por_edad.csv", index=False, encoding="utf-8-sig")
display(tablas["age"])


,Grupo,N,Tasa_Positivos_Real,Tasa_Positivos_Predicha,Recall,Precision,F1-Score,Positivos_Reales
0,18-29,2945,0.0499,0.0954,0.7143,0.3737,0.4907,147
1,30-39,2517,0.2706,0.3747,0.7959,0.5748,0.6675,681
2,40-49,2124,0.3719,0.5061,0.8620,0.6335,0.7303,790
3,50-59,1382,0.3734,0.5499,0.8527,0.5789,0.6897,516
4,60+,801,0.2547,0.5194,0.9265,0.4543,0.6097,204


## 4. Diferencias máximas entre grupos

La diferencia max-min resume el rango observado de cada indicador. No es una prueba estadística ni un umbral universal de equidad. La tasa positiva predicha aproxima paridad demográfica; recall aproxima igualdad de oportunidad para la etiqueta observada.


In [5]:
resumen = []
for atributo, tabla_grupo in tablas.items():
    etiqueta = "Edad" if atributo == "age" else ("Sexo" if atributo == "sex" else "Raza")
    resumen.append({
        "Atributo": etiqueta,
        "Brecha de recall (max-min)": diferencia_max_min(tabla_grupo, "Recall"),
        "Brecha de tasa positiva predicha (max-min)": diferencia_max_min(tabla_grupo, "Tasa_Positivos_Predicha"),
        "Menor tamaño de grupo": int(tabla_grupo["N"].min()),
    })
resumen_disparidad = pd.DataFrame(resumen)
display(resumen_disparidad)


,Atributo,Brecha de recall (max-min),Brecha de tasa positiva predicha (max-min),Menor tamaño de grupo
0,Sexo,0.1817,0.3258,3289
1,Raza,0.2126,0.2531,69
2,Edad,0.2122,0.4545,801


## 5. Consideraciones éticas

### Sesgos y límites

- **Sesgo histórico:** la fuente representa relaciones sociales y económicas de Estados Unidos en 1994, que no describen la actualidad ni otros países.
- **Representación desigual:** el número de observaciones varía mucho entre grupos, por lo que una métrica agregada oculta diferencias y los grupos pequeños tienen mayor incertidumbre.
- **Variables proxy:** ocupación, educación y estado civil pueden reflejar desigualdades asociadas con atributos sensibles.
- **Tasa base:** diferencias en ingresos observados afectan métricas de grupo; las tasas de positivos predichos no son directamente comparables sin contexto.
- **Privacidad:** aunque el conjunto es público y no contiene nombres, incluye atributos sensibles. La práctica debe limitarse al análisis agregado, evitar reidentificación y no publicar filas individuales.
- **Transparencia y justicia:** una predicción no explica una causa ni constituye una evaluación individual válida. Se requieren revisión humana, comunicación clara de límites, auditoría periódica y mecanismos de reclamación antes de cualquier aplicación.

### Uso responsable

El sistema es un ejercicio reproducible para aprender clasificación y auditoría. No debe desplegarse para decidir contratación, salarios, crédito, seguros, acceso a servicios ni oportunidades. Una mejora técnica de recall no elimina sesgos estructurales ni convierte el dataset en adecuado para esos usos.


## 6. Conclusión

Las métricas por grupo muestran que el desempeño y la tasa de predicción no son uniformes. Deben leerse junto con el tamaño de cada grupo, la antigüedad de los datos y el contexto social. El resultado respalda la recomendación de restringir el proyecto a fines académicos y mejorar la representatividad, la documentación y la evaluación antes de cualquier uso adicional.
